# SMiCA notebook: statistical microstructure characterisation, step by step

This notebook does the same calculations as the **SMiCA** app (Statistical Microstructure Characterisation & Analysis), but without the graphical interface: you give the path to your image(s), run the cells, and get the numbers, the plots and the saved files.

It is **stand-alone**. Every function it needs is written out in Part A, together with a short explanation of the method and references to the literature. Nothing is imported from the app.

## How the notebook is organised

| Part | Content |
|---|---|
| **0. Setup** | imports, output folder, optional demo data |
| **A. Functions** | the methods: $S_2$, $C_2$, lineal path, polytope functions, $\Omega$ metrics, chord length, REV/RES, Minkowski functionals, connected components |
| **1. 2D image** | one `.tif` image |
| **2. 3D volume** | one multi-page `.tif`, or a folder of 2D slices |
| **3. 2D + time** | a folder with one 2D image per time step |
| **4. 4D (3D + time)** | a folder with one 3D volume per time step |
| **References** | the papers and books cited in Part A |

## How to use it

1. Run **Part 0** and **Part A** once, from top to bottom. They only define things and take a few seconds.
2. Go to the part that matches your data, write your path in its first cell, and run the cells of that part. Parts 1 to 4 are independent of each other, and inside a part you can skip any analysis you do not need.

## Which analysis is available for which data

| Analysis | 2D | 3D | 2D + time | 4D |
|---|---|---|---|---|
| $S_2$, $C_2$, lineal path $L$ | yes | yes | every time step | every time step |
| Polytope functions $P_{3H}$, $P_{3V}$, $P_4$, $P_6$ | yes | no (2D only) | every time step | no |
| $\Omega$ and $\Delta\Omega$ evolution metrics | no | along the Z slices | yes | yes |
| Chord length | yes | yes | every time step | every time step |
| Representative size (RES in 2D, REV in 3D) | yes | yes | no | no |
| Minkowski functionals | yes | yes | every time step | every time step |
| Connected components | yes | yes | every frame labelled separately | no |
| Component measurements (size, shape) | yes | yes | no | no |

## Conventions

- **Images must be segmented.** All analyses work on a binary image: `1` is the phase you want to characterise (for example the pores) and `0` is everything else. If your image has several labels (0, 1, 2, ...), you choose which label is the foreground when you load it.
- **Array axes**: 2D `(Y, X)`, 3D `(Z, Y, X)`, 2D + time `(T, Y, X)`, 4D `(T, Z, Y, X)`.
- **Distances** $r$ are in pixels (voxels). Multiply by your pixel size to get physical lengths. The Minkowski functionals and the component measurements take the pixel size as an input and return physical units directly.

# Part 0. Setup

## Packages

The notebook needs the packages below. If one is missing, install it from a terminal (or put a `!` in front of the line and run it in a cell):

```
pip install "numpy<2" numba pandas matplotlib tifffile scikit-image scipy tqdm quantimpy
```

`quantimpy` is only used for the Minkowski functionals. It needs NumPy 1.x, which is why NumPy is pinned below 2.

In [ ]:
import os
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from numba import njit                     # compiles the pixel loops, so they run fast
from tqdm import tqdm                      # progress bar
from IPython.display import display        # shows tables nicely

from skimage.measure import label, regionprops, marching_cubes, mesh_surface_area

try:
    from quantimpy import minkowski as mk  # only needed for the Minkowski functionals
except Exception as error:
    mk = None
    print(f"quantimpy could not be imported ({error}).")
    print("Everything works except the Minkowski functionals.")

## Output settings

Every part saves its tables (`.csv`) and figures (`.png`) in its own sub-folder of `OUTPUT_DIR`.

In [ ]:
OUTPUT_DIR = 'results'     # folder for all saved tables and figures (created when needed)
CSV_SEP = ';'              # column separator in the saved .csv files (';' as in the app, use ',' if you prefer)

## Demo data (optional)

The cell below writes a small synthetic data set of each kind into the folder `demo_data`, so that the whole notebook can be run once before you use your own images. The images are smoothed random noise cut at a threshold, which gives blob-shaped "pores" (value 1). In the two time series the threshold rises with time, so the pores grow.

Skip this cell if you only work with your own data.

In [ ]:
from scipy.ndimage import gaussian_filter


def make_demo_data(folder='demo_data', seed=0):
    """Write one small synthetic example of each kind of data into `folder`."""
    rng = np.random.default_rng(seed)

    def blobs(shape, porosities, sigma=4):
        # smooth random noise, then call the lowest values "pore": one image per requested porosity
        field = gaussian_filter(rng.standard_normal(shape), sigma, mode='wrap')
        return [(field < np.quantile(field, p)).astype(np.uint8) for p in porosities]

    os.makedirs(os.path.join(folder, 'time_series_2d'), exist_ok=True)
    os.makedirs(os.path.join(folder, 'time_series_3d'), exist_ok=True)

    tifffile.imwrite(os.path.join(folder, 'image_2d.tif'), blobs((256, 256), [0.25])[0])
    tifffile.imwrite(os.path.join(folder, 'volume_3d.tif'), blobs((96, 96, 96), [0.25])[0])
    for t, frame in enumerate(blobs((256, 256), np.linspace(0.10, 0.35, 8))):
        tifffile.imwrite(os.path.join(folder, 'time_series_2d', f'frame_{t:03d}.tif'), frame)
    for t, volume in enumerate(blobs((64, 64, 64), np.linspace(0.10, 0.35, 5))):
        tifffile.imwrite(os.path.join(folder, 'time_series_3d', f'volume_{t:03d}.tif'), volume)
    print('demo data written to', os.path.abspath(folder))


make_demo_data()

# Part A. Functions

Run all cells of this part once. Each section first explains the method, then defines the functions. The analysis parts (1 to 4) only call these functions.

A note on speed: the functions marked with `@njit` are compiled by Numba (Lam et al., 2015) the first time they are called. That first call takes a few seconds; after that the pixel loops run about as fast as C code.

## A1. Reading, checking and saving

- `load_image` reads one `.tif` file: a normal image gives a 2D array, a multi-page file gives a 3D array.
- `list_files_sorted` lists the `.tif` files of a folder, sorted by a number in the file name (slice number or time step). File names often contain several numbers (`sample7_scan2_t015.tif`), so you can choose which one counts: `number_position=-1` is the last number, `0` the first, `1` the second, and so on.
- `load_stack` reads several files and stacks them: 2D slices become a 3D volume or a 2D time series, 3D volumes become a 4D array.
- `binarize` turns one label of a segmented image into the foreground (1) and everything else into background (0).

In [ ]:
def load_image(path):
    """Read one TIF file: a 2D array (Y, X), or a 3D array (Z, Y, X) for a multi-page file."""
    image = tifffile.imread(path)
    if image.ndim == 3 and image.shape[-1] in (3, 4):    # colour image (RGB or RGBA): keep one channel
        image = image[..., 0]
    if image.ndim == 3 and image.shape[0] == 1:          # a "stack" with a single page is a 2D image
        image = image[0]
    return image


def list_files_sorted(folder, number_position=-1):
    """All .tif files of a folder, sorted by a number in the file name.

    Returns (paths, numbers): the full paths in sorted order, and the number found in each name.
    """
    names = [f for f in os.listdir(folder) if f.lower().endswith(('.tif', '.tiff'))]
    numbers = [int(re.findall(r'\d+', name)[number_position]) for name in names]
    pairs = sorted(zip(numbers, names))
    paths = [os.path.join(folder, name) for _, name in pairs]
    return paths, [number for number, _ in pairs]


def load_stack(paths):
    """Read several files of the same shape and stack them along a new first axis."""
    return np.stack([load_image(path) for path in paths])


def binarize(image, foreground_value=1):
    """Binary image: 1 where the image equals `foreground_value`, 0 everywhere else."""
    return (image == foreground_value).astype(np.uint8)


def describe(image, name='image'):
    """Print the size of an image and the pixel values it contains."""
    values = np.unique(image)
    print(f"{name}: shape {image.shape}, type {image.dtype}, pixel values {values[:10]}")
    if len(values) > 2:
        print("   more than two values: choose the foreground label with binarize(image, foreground_value)")
    else:
        print(f"   fraction of pixels equal to {values[-1]}: {(image == values[-1]).mean():.4f}")


def show_image(image, title=''):
    """Show a 2D image, or the three orthogonal slices through the centre of a 3D volume."""
    if image.ndim == 2:
        views, names = [image], [title]
    else:
        z, y, x = np.array(image.shape) // 2
        views = [image[z], image[:, y, :], image[:, :, x]]
        names = [f'{title}  XY plane (z={z})', f'XZ plane (y={y})', f'YZ plane (x={x})']
    fig, axes = plt.subplots(1, len(views), figsize=(4.5 * len(views), 4.5), squeeze=False)
    for ax, view, name in zip(axes[0], views, names):
        ax.imshow(view, cmap='gray', interpolation='nearest')
        ax.set_title(name)
        ax.axis('off')
    plt.show()


def save_table(table, path):
    """Save a pandas table as a .csv file."""
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    table.to_csv(path, sep=CSV_SEP, index=False)
    print('saved', path)


# Plot style, the same as in the app (src/micro_gui/gui/plot_style.py). Sized for figures of
# PANEL_W x PANEL_H inches per panel that are shrunk to 50% in a paper, so every printed figure
# has axis labels of 9 pt, tick labels of 8 pt and legends of 7 pt.
PANEL_W, PANEL_H = 7.0, 5.0
plt.rcParams.update({
    'font.size': 16,
    'axes.labelsize': 18,           # x, y and colorbar labels
    'xtick.labelsize': 16,          # tick labels
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
    'axes.titlesize': 16,           # titles are shown here but left out of saved files (_finish)
    'axes.titleweight': 'bold',
    'lines.linewidth': 2.0,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2, 'ytick.major.width': 1.2,
    'xtick.major.size': 6.0, 'ytick.major.size': 6.0,
    'xtick.minor.width': 0.9, 'ytick.minor.width': 0.9,
    'xtick.minor.size': 3.5, 'ytick.minor.size': 3.5,
    'figure.constrained_layout.use': True,
})

# When the phase is one big connected cluster, C2 = S2 and the two curves coincide: S2 is drawn
# twice as wide and underneath the other curves, so it shows as a border around C2.
CURVE_STYLES = {'s2': {'linewidth': 4.0, 'zorder': 1.9}}


def mark_phase_fraction(ax, phi):
    """Black circle at (r = 0, phi) on a raw P_n(r) plot: every raw SMD starts at the phase fraction.
    No legend entry. Not for the scaled f_n(r) curves, which always start at 1."""
    ax.plot([0], [phi], linestyle='none', marker='o', markersize=8, color='black', zorder=3, clip_on=False,
            label='_nolegend_')


def r_label(ndim):
    """x-axis label for curves against the distance r (r in italic, as in $P_n(r)$):
    pixels for 2D images, voxels for 3D volumes."""
    return '$r$ (pixels)' if ndim == 2 else '$r$ (voxels)'


def _finish(fig, save_as=None):
    """Used by all plot functions: save the figure if a file name is given, then show it.
    As in the app, the saved file has no titles; they stay visible in the notebook."""
    if save_as:
        os.makedirs(os.path.dirname(save_as) or '.', exist_ok=True)
        suptitle = getattr(fig, '_suptitle', None)            # set by fig.suptitle(), if used
        titles = [ax.title for ax in fig.axes] + ([suptitle] if suptitle is not None else [])
        titles = [title for title in titles if title.get_visible()]
        for title in titles:
            title.set_visible(False)
        fig.savefig(save_as, dpi=500, bbox_inches='tight')
        for title in titles:
            title.set_visible(True)
        print('saved', save_as)
    plt.show()

## A2. Two-point probability function $S_2(r)$

**What it is.** Throw a line segment of length $r$ at random into the image. $S_2(r)$ is the probability that *both ends* land in the phase of interest:

$$S_2(r) = \big\langle I(\mathbf{x})\, I(\mathbf{x}+\mathbf{r}) \big\rangle , \qquad I(\mathbf{x}) = 1 \text{ inside the phase and } 0 \text{ outside.}$$

**How to read the curve.**
- $S_2(0) = \phi$, the volume fraction (porosity), because both ends are the same point.
- At large $r$ the two ends are independent, so $S_2 \to \phi^2$ (when there is no long-range order).
- The distance over which the curve falls from $\phi$ to $\phi^2$ is the typical size of the features.
- The slope at $r = 0$ is set by the specific surface area $s$: for an isotropic medium $dS_2/dr = -s/4$ in 3D and $-s/\pi$ in 2D (Debye et al., 1957; Berryman & Blair, 1986).
- Oscillations mean that the features are regularly spaced.

**Scaled form.** To compare images with different porosity, the curve is rescaled so that it starts at 1 and decays to 0 (the scaled autocovariance):

$$f_2(r) = \frac{S_2(r) - \phi^2}{\phi - \phi^2}$$

The function `cal_fn` does this for any number of points $n$, with $\phi^n$ in place of $\phi^2$. It is used again for the polytope functions in A5. The curve is not forced to reach 0: a tail above 0 means the structure is still correlated at that distance, and small negative values are normal noise.

**How it is computed.** Following Torquato (2002, Sec. 12.5.1) and Yeong & Torquato (1998): take every row and every column of the image (and every line along Z in 3D). For each distance $r$, count the pixel pairs $(i,\, i+r)$ on that line that are both in the phase, then divide by the number of pairs tried. The image is treated as **periodic**: a segment that leaves on one side re-enters on the opposite side, so every pixel is a valid starting point for every $r$. The distance runs from 0 to half of the smallest image size, because beyond that a periodic image only repeats itself. Only the axis directions are sampled and they are averaged, so the result describes the structure as if it were isotropic.

**About the code.** `_pair_counts_line` counts the pairs of pixels that carry the *same non-zero value*. On a binary image this gives $S_2$. On an image in which every pixel holds the number of its cluster, the very same counting gives the cluster function $C_2$ of the next section.

In [ ]:
@njit
def _pair_counts_line(line, n, nt):
    """One line of n pixels with periodic ends. For every distance r = 0 .. nt-1, count the
    pairs of pixels (i, i + r) that carry the same non-zero value."""
    counts = np.zeros(nt)
    for i in range(n):
        value = line[i]
        if value == 0:                 # background pixel: never part of a pair
            continue
        for r in range(nt):
            j = i + r
            if j >= n:                 # periodic boundary: past the end, continue from the start
                j -= n
            if line[j] == value:
                counts[r] += 1.0
    return counts


@njit
def _pair_probability_2d(image, nt):
    """Pair counts along every row and every column, divided by the number of pairs tried."""
    n_rows, n_cols = image.shape
    total = np.zeros(nt)
    for row in range(n_rows):
        total += _pair_counts_line(image[row, :], n_cols, nt)
    for col in range(n_cols):
        total += _pair_counts_line(image[:, col], n_rows, nt)
    return total / (2.0 * n_rows * n_cols)       # 2 directions, n_rows * n_cols starting pixels each


@njit
def _pair_probability_3d(image, nt):
    """Same as the 2D version, along the lines of all three axes."""
    n0, n1, n2 = image.shape
    total = np.zeros(nt)
    for i in range(n1):
        for j in range(n2):
            total += _pair_counts_line(image[:, i, j], n0, nt)     # lines along Z
    for i in range(n0):
        for j in range(n2):
            total += _pair_counts_line(image[i, :, j], n1, nt)     # lines along Y
    for i in range(n0):
        for j in range(n1):
            total += _pair_counts_line(image[i, j, :], n2, nt)     # lines along X
    return total / (3.0 * n0 * n1 * n2)


def calculate_s2_periodic(image):
    """S2(r) of a 2D binary image, r = 0 .. min(image.shape) // 2 - 1. The image need not be square."""
    binary = (image != 0).astype(np.uint8)
    return _pair_probability_2d(binary, min(image.shape) // 2)


def calculate_s2_periodic_3d(image):
    """S2(r) of a 3D binary volume, r = 0 .. min(image.shape) // 2 - 1. The volume need not be cubic."""
    binary = (image != 0).astype(np.uint8)
    return _pair_probability_3d(binary, min(image.shape) // 2)


def cal_fn(polytope, n):
    """Scaled n-point function: f_n(r) = (P_n(r) - phi^n) / (phi - phi^n), with phi = P_n(0).

    n is the number of points: 2 for S2, 3 for the triangles, 4 for the square, 6 for the hexagon.
    If the phase is absent (phi = 0) or fills the whole image (phi = 1), f_n is undefined: all NaN.
    """
    phi = polytope[0]
    if phi - phi ** n == 0:                    # phi = 0 or 1: no two-phase structure (0/0)
        return np.full(np.shape(polytope), np.nan)
    return (polytope - phi ** n) / (phi - phi ** n)

## A3. Two-point cluster function $C_2(r)$

**What it is.** $C_2(r)$ is the probability that the two ends of a segment of length $r$ land in the **same cluster** of the phase, meaning there is a path from one end to the other that never leaves the phase (Torquato et al., 1988). $S_2$ only asks whether both ends are in the phase; $C_2$ also asks whether they are connected. This makes $C_2$ sensitive to connectivity and percolation, information that $S_2$ does not contain (Jiao et al., 2009).

**How to read the curve.**
- $C_2(0) = \phi$, and $C_2(r) \le S_2(r)$ for every $r$.
- If the phase consists of isolated clusters, $C_2$ drops to 0 once $r$ exceeds the largest cluster.
- If one cluster spans the whole image (a percolating phase), $C_2$ stays above 0 at large $r$.

**Scaled form.** $C_2(r) / S_2(r)$ is the probability that two points are connected, *given* that both are in the phase. It starts at 1 and stays between 0 and 1. A value near 1 means the phase is well connected over that distance. Where $S_2(r) = 0$ the ratio is undefined and is returned as `NaN`.

**How it is computed.**
1. *Label the clusters.* Every pixel of the phase gets the number of the cluster it belongs to. Two pixels belong to the same cluster when they share a face (4 neighbours in 2D, 6 in 3D). The image is periodic here as well, so a cluster that touches one side continues on the opposite side. The labelling is a "flood fill": start from an unlabelled pixel, give it a new number, then keep visiting its unlabelled neighbours until the cluster is complete.
2. *Count pairs with the same label*, exactly as for $S_2$, with `_pair_probability_2d` / `_pair_probability_3d` from A2.

For a large 3D volume (512³ and more) the labelling needs a few GB of memory and the calculation can take minutes.

In [ ]:
@njit
def _label_clusters_periodic(binary):
    """Number the clusters of a 3D binary array (face neighbours, periodic in all directions).

    Returns an array of the same shape: 0 = background, 1, 2, 3, ... = cluster number.
    A 2D image is passed as a volume with a single slice.
    """
    n0, n1, n2 = binary.shape
    labels = np.zeros((n0, n1, n2), dtype=np.int32)
    stack = np.empty((n0 * n1 * n2, 3), dtype=np.int32)     # to-do list: voxels whose neighbours must be checked
    current = 0

    for s0 in range(n0):
        for s1 in range(n1):
            for s2 in range(n2):
                if binary[s0, s1, s2] == 1 and labels[s0, s1, s2] == 0:
                    current += 1                            # an unlabelled phase voxel: a new cluster starts here
                    labels[s0, s1, s2] = current
                    stack[0, 0], stack[0, 1], stack[0, 2] = s0, s1, s2
                    size = 1
                    while size > 0:                         # flood fill
                        size -= 1
                        a, b, c = stack[size, 0], stack[size, 1], stack[size, 2]
                        for axis in range(3):
                            for step in (-1, 1):
                                na, nb, nc = a, b, c
                                if axis == 0:
                                    na = (a + step) % n0    # % wraps around the image edge (periodic)
                                elif axis == 1:
                                    nb = (b + step) % n1
                                else:
                                    nc = (c + step) % n2
                                if binary[na, nb, nc] == 1 and labels[na, nb, nc] == 0:
                                    labels[na, nb, nc] = current
                                    stack[size, 0], stack[size, 1], stack[size, 2] = na, nb, nc
                                    size += 1
    return labels


def calculate_c2(image):
    """C2(r) of a 2D binary image, r = 0 .. min(image.shape) // 2 - 1."""
    binary = (image != 0).astype(np.uint8)
    labels = _label_clusters_periodic(binary[np.newaxis])[0]     # label the image as a one-slice volume
    return _pair_probability_2d(labels, min(image.shape) // 2)


def calculate_c2_3d(image):
    """C2(r) of a 3D binary volume, r = 0 .. min(image.shape) // 2 - 1."""
    binary = (image != 0).astype(np.uint8)
    labels = _label_clusters_periodic(binary)
    return _pair_probability_3d(labels, min(image.shape) // 2)


def scale_c2_by_connectedness(c2_values, s2_values):
    """C2(r) / S2(r): probability of being connected, given that both points are in the phase."""
    with np.errstate(invalid='ignore', divide='ignore'):     # S2(r) = 0 gives NaN, without a warning
        return c2_values / s2_values

## A4. Lineal-path function $L(r)$

**What it is.** $L(r)$ is the probability that a **whole** segment of length $r$ lies inside the phase, not only its two ends (Lu & Torquato, 1992). It measures how far you can travel in a straight line without leaving the phase.

**How to read the curve.**
- $L(0) = \phi$, and $L(r) \le S_2(r)$.
- $L$ decays to 0: once $r$ is longer than the longest straight run of the phase, no segment fits any more.
- A slow decay means long straight channels or large grains.

**Scaled form.** Since $L$ goes to 0 by itself, the scaled curve is simply $L(r)/L(0)$.

**How it is computed.** Along one line of pixels, the phase appears as unbroken runs called **chords**. A chord of length $c$ pixels has room for $c - r$ segments of length $r$. So for every chord, $c - r$ is added to each bin $r < c$, and the total is divided by the number of starting positions. Lines are again periodic and are scanned along every axis.

In [ ]:
@njit
def _chord_counts_line(line, n, nt):
    """One periodic line of n pixels. For every distance r = 0 .. nt-1, count the positions
    where a segment of length r lies completely inside the phase."""
    counts = np.zeros(nt)

    start = -1
    for i in range(n):                 # find a background pixel to start the walk from,
        if line[i] == 0:               # so that no chord is cut in two by the start of the line
            start = i
            break
    if start == -1:                    # the whole periodic line is phase:
        counts[:] = n                  # a segment of any length fits at each of the n positions
        return counts

    run = 0                            # length of the chord we are currently walking through
    for k in range(1, n + 1):          # walk once around the line, ending on the start pixel
        i = start + k
        if i >= n:
            i -= n
        if line[i] == 1:
            run += 1
        elif run > 0:                  # a chord has just ended
            for r in range(min(run, nt)):
                counts[r] += run - r   # a chord of length c holds (c - r) segments of length r
            run = 0
    return counts


@njit
def _lineal_path_2d(binary, nt):
    n_rows, n_cols = binary.shape
    total = np.zeros(nt)
    for row in range(n_rows):
        total += _chord_counts_line(binary[row, :], n_cols, nt)
    for col in range(n_cols):
        total += _chord_counts_line(binary[:, col], n_rows, nt)
    return total / (2.0 * n_rows * n_cols)


@njit
def _lineal_path_3d(binary, nt):
    n0, n1, n2 = binary.shape
    total = np.zeros(nt)
    for i in range(n1):
        for j in range(n2):
            total += _chord_counts_line(binary[:, i, j], n0, nt)
    for i in range(n0):
        for j in range(n2):
            total += _chord_counts_line(binary[i, :, j], n1, nt)
    for i in range(n0):
        for j in range(n1):
            total += _chord_counts_line(binary[i, j, :], n2, nt)
    return total / (3.0 * n0 * n1 * n2)


def calculate_L_3d(image):
    """Lineal-path function L(r) of a 3D binary volume, r = 0 .. min(image.shape) // 2 - 1."""
    binary = (image != 0).astype(np.uint8)
    return _lineal_path_3d(binary, min(image.shape) // 2)


def scale_by_initial_value(values):
    """f(r) = values(r) / values(0). Used for L, which decays to 0 on its own.
    If the phase is absent (L(0) = 0), the result is all NaN."""
    if values[0] == 0:
        return np.full(np.shape(values), np.nan)
    return values / values[0]

## A5. n-point polytope functions $P_n(r)$ (2D only)

**What they are.** $S_2$ uses two points. The polytope functions of Chen et al. (2019) use more: $P_n(r)$ is the probability that **all $n$ corners** of a regular polygon with edge length $r$ land in the phase. Each one "sees" a different part of the geometry that $S_2$ misses.

| Function | Shape | $n$ |
|---|---|---|
| `p3h`, `p3v` | equilateral triangle, with one edge horizontal (`p3h`) or vertical (`p3v`) | 3 |
| `p4` | square | 4 |
| `p6` | regular hexagon | 6 |

**How to read the curves.** $P_n(0) = \phi$ (all corners on the same pixel) and $P_n \to \phi^n$ at large $r$ (independent corners). The scaled form is

$$f_n(r) = \frac{P_n(r) - \phi^n}{\phi - \phi^n}$$

which runs from 1 towards 0 like $f_2$, so all functions can be compared in one plot.

**How they are computed.** Every phase pixel is used as the first corner. The other corners are placed at the right offsets (rounded to whole pixels) and the polygon counts when all corners are in the phase. Three details of the implementation (the same as in the app):

- The image does **not** need to be square. Like $S_2$ and $C_2$, it wraps around at its own size along each axis (a row at the number of columns, a column at the number of rows), so $P_n(0)$ is exactly the porosity. Every curve runs from $r = 0$ to $\min(\text{shape}) // 2 - 1$, the largest distance that fits in both directions (e.g. $r = 0 \dots 274$ for a $551 \times 580$ image).
- The C++ reference code that the app was validated against only takes square images, which it pads with one row and one column of background (period $N + 1$). Padding the image the same way first, `np.pad(image, ((0, 1), (0, 1)))`, reproduces its results.
- The triangles are only evaluated at **even** $r$, where half the edge is a whole number of pixels. The odd entries are left at 0 and must be skipped when plotting (the plot functions below do this).

In [ ]:
def _binary_config(image):
    """0/1 version of a 2D image and the number of distances nt = min(image.shape) // 2.

    No padding: rows wrap around at the number of columns and columns at the number of rows,
    so the image does not need to be square. Returns (config, nt).
    """
    if image.ndim != 2:
        raise ValueError('The polytope functions (L, p3h, p3v, p4, p6) need a 2D image.')
    config = (image != 0).astype(np.uint8)
    return config, min(config.shape) // 2


@njit
def compute_p4_polytope(config, nt):
    """P4: squares with corners (r, c), (r, c+R), (r+R, c), (r+R, c+R)."""
    n_rows, n_cols = config.shape
    counts = np.zeros(nt)
    for R in range(nt):
        for r in range(n_rows):
            for c in range(n_cols):
                if config[r, c] == 1:
                    xx = (c + R) % n_cols                      # a column index wraps at the number of columns,
                    yy = (r + R) % n_rows                      # a row index at the number of rows
                    if config[r, xx] == 1 and config[yy, c] == 1 and config[yy, xx] == 1:
                        counts[R] += 1.0
    return counts / (n_rows * n_cols)                          # every pixel is tried once as the first corner


@njit
def compute_p3v_polytope(config, nt):
    """P3V: triangles with a vertical edge of length R, corners (r, c), (r+R, c) and the tip
    half-way down, R/2 * sqrt(3) pixels to the side. Even R only."""
    n_rows, n_cols = config.shape
    counts = np.zeros(nt)
    sqrt3 = math.sqrt(3.0)
    for R in range(0, nt, 2):
        half = R // 2
        shift = 0 if R == 0 else int(half * sqrt3 + 1.0)       # height of the triangle, in whole pixels
        for r in range(n_rows):
            for c in range(n_cols):
                if config[r, c] == 1:
                    xx = (c + shift) % n_cols
                    gg = (r + half) % n_rows
                    yy = (r + R) % n_rows
                    if config[yy, c] == 1 and config[gg, xx] == 1:
                        counts[R] += 1.0
    return counts / (n_rows * n_cols)


@njit
def compute_p3h_polytope(config, nt):
    """P3H: the same triangle with a horizontal edge (rows and columns swapped). Even R only."""
    n_rows, n_cols = config.shape
    counts = np.zeros(nt)
    sqrt3 = math.sqrt(3.0)
    for R in range(0, nt, 2):
        half = R // 2
        shift = 0 if R == 0 else int(half * sqrt3 + 1.0)
        for r in range(n_rows):
            for c in range(n_cols):
                if config[r, c] == 1:
                    yy = (r + shift) % n_rows
                    gg = (c + half) % n_cols
                    xx = (c + R) % n_cols
                    if config[r, xx] == 1 and config[yy, gg] == 1:
                        counts[R] += 1.0
    return counts / (n_rows * n_cols)


@njit
def compute_p6_polytope(config, nt):
    """P6: regular hexagons with edge length R. (r, c) is the first corner, the other five
    corners are placed around it and rounded up to whole pixels (x = row, y = column)."""
    n_rows, n_cols = config.shape
    counts = np.zeros(nt)
    sqrt3 = math.sqrt(3.0)
    for R in range(nt):
        h = sqrt3 * 0.5 * R                                    # half the width of the hexagon
        for r in range(n_rows):
            for c in range(n_cols):
                if config[r, c] == 1:
                    x2 = math.ceil(r + h) % n_rows
                    y2 = math.ceil(c - 0.5 * R) % n_cols       # % also brings a negative column back into range
                    x3 = math.ceil(r + h + h) % n_rows
                    y3 = c
                    x4 = x3
                    y4 = (c + R) % n_cols
                    x5 = x2
                    y5 = math.ceil(c + R + 0.5 * R) % n_cols
                    x6 = r
                    y6 = (c + R) % n_cols
                    if (config[x2, y2] == 1 and config[x3, y3] == 1 and config[x4, y4] == 1
                            and config[x5, y5] == 1 and config[x6, y6] == 1):
                        counts[R] += 1.0
    return counts / (n_rows * n_cols)

## A6. One call for all descriptors

`compute_2d_polytope_curves` and `compute_3d_polytope_curves` compute the descriptors you ask for on one image and return two tables, **raw** and **scaled**. Each table has a column `r` and one column per descriptor.

| Name | Descriptor | Scaled as | 2D | 3D |
|---|---|---|---|---|
| `'s2'` | two-point probability | $(S_2-\phi^2)/(\phi-\phi^2)$ | yes | yes |
| `'c2'` | two-point cluster function | $C_2/S_2$ | yes | yes |
| `'L'` | lineal path | $L/L(0)$ | yes | yes |
| `'p3h'`, `'p3v'` | triangles | $(P_3-\phi^3)/(\phi-\phi^3)$ | yes | no |
| `'p4'` | square | $(P_4-\phi^4)/(\phi-\phi^4)$ | yes | no |
| `'p6'` | hexagon | $(P_6-\phi^6)/(\phi-\phi^6)$ | yes | no |

In [ ]:
POLYTOPES_2D = {                      # name: (function, number of corners)
    'p3h': (compute_p3h_polytope, 3),
    'p3v': (compute_p3v_polytope, 3),
    'p4': (compute_p4_polytope, 4),
    'p6': (compute_p6_polytope, 6),
}


def _curves_to_table(curves, selected):
    """Dictionary of curves -> table with a column 'r' and one column per selected function."""
    table = pd.DataFrame({name: curves[name] for name in selected})
    table.insert(0, 'r', np.arange(len(table)))
    return table


def compute_2d_polytope_curves(image, selected):
    """Raw and scaled curves of ONE 2D binary image.

    selected: list with any of 's2', 'c2', 'L', 'p3h', 'p3v', 'p4', 'p6'.
    Returns (raw, scaled), two tables with a column 'r' and one column per selected function.
    """
    unknown = set(selected) - {'s2', 'c2', 'L', 'p3h', 'p3v', 'p4', 'p6'}
    if unknown:
        raise ValueError(f"Unknown function(s) {unknown}. Use 's2', 'c2', 'L', 'p3h', 'p3v', 'p4', 'p6'.")
    raw, scaled = {}, {}

    if 's2' in selected or 'c2' in selected:           # S2 is also needed to scale C2
        s2 = calculate_s2_periodic(image)
        raw['s2'], scaled['s2'] = s2, cal_fn(s2, n=2)
    if 'c2' in selected:
        raw['c2'] = calculate_c2(image)
        scaled['c2'] = scale_c2_by_connectedness(raw['c2'], s2)

    if set(selected) - {'s2', 'c2'}:                   # L and the polytopes: same r range as S2 and C2
        config, nt = _binary_config(image)
    if 'L' in selected:
        raw['L'] = _lineal_path_2d(config, nt)
        scaled['L'] = scale_by_initial_value(raw['L'])
    for name, (function, n_corners) in POLYTOPES_2D.items():
        if name in selected:
            raw[name] = function(config, nt)
            scaled[name] = cal_fn(raw[name], n=n_corners)

    return _curves_to_table(raw, selected), _curves_to_table(scaled, selected)


def compute_3d_polytope_curves(image, selected):
    """Raw and scaled curves of ONE 3D binary volume. selected: list with any of 's2', 'c2', 'L'."""
    unknown = set(selected) - {'s2', 'c2', 'L'}
    if unknown:
        raise ValueError(f"{unknown} not available in 3D. Use 's2', 'c2', 'L'.")
    raw, scaled = {}, {}

    if 's2' in selected or 'c2' in selected:
        s2 = calculate_s2_periodic_3d(image)
        raw['s2'], scaled['s2'] = s2, cal_fn(s2, n=2)
    if 'c2' in selected:
        raw['c2'] = calculate_c2_3d(image)
        scaled['c2'] = scale_c2_by_connectedness(raw['c2'], s2)
    if 'L' in selected:
        raw['L'] = calculate_L_3d(image)
        scaled['L'] = scale_by_initial_value(raw['L'])

    return _curves_to_table(raw, selected), _curves_to_table(scaled, selected)


def plot_curves(raw, scaled, title='', save_as=None, ndim=2):
    """Raw curves (left) and scaled curves (right), one line per function.
    ndim: 2 for curves of a 2D image (r in pixels), 3 for a 3D volume (r in voxels)."""
    fig, axes = plt.subplots(1, 2, figsize=(2 * PANEL_W, PANEL_H))
    for ax, table, ylabel in zip(axes, (raw, scaled), ('raw  $P_n(r)$', 'scaled  $f_n(r)$')):
        for name in table.columns.drop('r'):
            skip = 2 if name in ('p3h', 'p3v') else 1          # the triangles exist at even r only
            ax.plot(table['r'][::skip], table[name][::skip], label=name.upper(), **CURVE_STYLES.get(name, {}))
        if table is raw:                                   # raw curves start at phi, scaled ones at 1
            mark_phase_fraction(ax, table.iloc[0, 1])
        ax.set_xlabel(r_label(ndim))
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(title)
    _finish(fig, save_as)

## A7. Evolution metrics $\Omega$ and $\Delta\Omega$

When you have a series of images (time steps, or the slices of a volume), you get one curve per step and per descriptor. To follow the change with a single number per step, Chen et al. (2022) proposed the $\Omega$ metrics. For a descriptor $P$ with $N_r$ values of $r$:

$$\Omega(t) = \frac{1}{N_r}\sum_r \big| P(r, t) - P(r, t_\mathrm{ref}) \big| \qquad\qquad \Delta\Omega(t) = \frac{1}{N_r}\sum_r \big| P(r, t) - P(r, t-1) \big|$$

- $\Omega(t)$ is the distance to **one reference step** (usually the first). It shows how far the structure has moved away from where it started.
- $\Delta\Omega(t)$ is the distance to the **previous step**. It shows how fast the structure is changing at that moment. It is 0 for the first step.
- With `signed=True`, $\Delta\Omega$ uses the plain difference instead of the absolute value. It then has a sign (net increase or decrease), but increases and decreases at different $r$ can cancel each other.

Both are computed from the **raw** curves.

In [ ]:
def omega_n(curves, reference_index=0):
    """Omega: mean absolute difference between each curve and the reference curve.

    curves: list of 1D arrays of the same length, one per step. Returns one value per step.
    """
    curves = np.nan_to_num(np.array(curves))                   # shape (number of steps, number of r)
    return np.abs(curves - curves[reference_index]).sum(axis=1) / curves.shape[1]


def delta_omega(curves, signed=False):
    """Delta-Omega: mean absolute difference between each curve and the previous one (0 for the first)."""
    curves = np.nan_to_num(np.array(curves))
    difference = np.diff(curves, axis=0)                       # curve(t) - curve(t-1)
    change = difference.sum(axis=1) if signed else np.abs(difference).sum(axis=1)
    return np.concatenate(([0.0], change / curves.shape[1]))


def evolution_metrics(raw_tables, steps, reference_index=0, signed=False):
    """Omega and Delta-Omega of every function in a series of raw tables (one table per step).

    Returns a table with a column 'step' and the columns 'omega_<name>' and 'delta_omega_<name>'.
    """
    table = pd.DataFrame({'step': steps})
    for name in raw_tables[0].columns.drop('r'):
        skip = 2 if name in ('p3h', 'p3v') else 1              # leave out the empty odd-r entries
        curves = [raw[name].to_numpy()[::skip] for raw in raw_tables]
        table[f'omega_{name}'] = omega_n(curves, reference_index)
        table[f'delta_omega_{name}'] = delta_omega(curves, signed)
    return table

## A8. Chord length

**What it is.** Draw a straight line through the image. The pieces of the line that lie inside the phase are **chords**. Their mean length $\ell_c$ is a direct measure of feature size (pore or grain size along a line), and their distribution $p(z)$ shows how variable that size is (Torquato & Lu, 1993; Torquato, 2002, Ch. 2).

The mean chord length is computed in two independent ways, which should agree:

**Method 1, from the slope of $S_2$ at the origin:**

$$\ell_c = \frac{\phi}{\left| dS_2/dr \right|_{r=0}}$$

The reason is simple on a line of pixels. $S_2(0) - S_2(1)$ is the fraction of pixels that are in the phase while their next neighbour is not, which is the number of chord ends per pixel. The phase fraction divided by the number of chords per pixel is the mean chord length. The slope is taken from the first `num_points` values of $S_2$. Keep `num_points = 2`: real $S_2$ curves bend near the origin, so a wider fit window underestimates the slope and overestimates $\ell_c$.

**Method 2, direct measurement.** Scan every line of the image along every axis and measure the length of each unbroken run of phase pixels. Runs that touch the image border are discarded, because their true length is unknown. This method also gives the full distribution $p(z)$, and the mean is $\ell_c = \langle z \rangle$.

Small differences between the two values are expected: method 1 uses the periodic $S_2$, method 2 drops the border chords.

In [ ]:
def chord_length_from_S2(s2, num_points=2, dr=1.0):
    """Mean chord length from the slope of S2 at r = 0:  phi / |dS2/dr|."""
    phi = s2[0]                                    # S2(0) is the volume fraction
    n = min(num_points, len(s2))
    r = np.arange(n) * dr
    slope = np.polyfit(r, s2[:n], 1)[0]            # straight-line fit through the first n points
    return phi / abs(slope)


def _runs_1d(line):
    """Lengths of the runs of 1s in a 1D binary array, without the runs that touch either end."""
    padded = np.concatenate(([0], line, [0]))
    change = np.diff(padded)                       # +1 where a run starts, -1 just after it ends
    starts = np.where(change == 1)[0]
    ends = np.where(change == -1)[0]
    interior = (starts > 0) & (ends < len(line))   # drop the runs cut by the border
    return (ends - starts)[interior].tolist()


def sample_chords_nd(image, phase=1):
    """All chord lengths of a 2D or 3D image, measured along every axis."""
    binary = (image == phase).astype(np.uint8)
    chords = []
    for axis in range(binary.ndim):
        lines = np.moveaxis(binary, axis, 0).reshape(binary.shape[axis], -1)   # each column is one scan line
        for line in lines.T:
            chords += _runs_1d(line)
    return np.asarray(chords, dtype=float)


def chord_pdf(chords, nbins=40):
    """Chord-length distribution: bin centres z, probability density p(z), and the mean chord length."""
    p_z, edges = np.histogram(chords, bins=nbins, density=True)
    z = 0.5 * (edges[:-1] + edges[1:])
    return z, p_z, chords.mean()


def compute_chord_length_result(image, num_points=2, dr=1.0, nbins=40):
    """Mean chord length (both methods) and chord-length distribution of phase 1, in 2D or 3D.

    Returns a dictionary: 'ell_via_s2', 'ell_direct', 'chords' (all chord lengths), 'z', 'p_z'.
    """
    s2 = calculate_s2_periodic(image) if image.ndim == 2 else calculate_s2_periodic_3d(image)
    chords = sample_chords_nd(image, phase=1)
    z, p_z, ell_direct = chord_pdf(chords, nbins=nbins)
    return {
        'ell_via_s2': chord_length_from_S2(s2, num_points=num_points, dr=dr),   # method 1
        'ell_direct': ell_direct,                                               # method 2
        'chords': chords,
        'z': z,
        'p_z': p_z,
    }


def plot_chord(result, unit='pixels', save_as=None):
    """Chord-length distribution with the two mean values marked."""
    fig, ax = plt.subplots(figsize=(PANEL_W, PANEL_H))
    ax.bar(result['z'], result['p_z'], width=result['z'][1] - result['z'][0], alpha=0.7, label='chord-length distribution')
    ax.axvline(result['ell_via_s2'], color='red', ls='--', lw=2, label=f"mean via S2 = {result['ell_via_s2']:.2f}")
    ax.axvline(result['ell_direct'], color='black', ls='--', lw=2, label=f"mean, direct = {result['ell_direct']:.2f}")
    ax.set_xlabel(f'chord length ({unit})')
    ax.set_ylabel('$p(z)$')
    ax.grid(alpha=0.3)
    ax.legend()
    _finish(fig, save_as)

## A9. Representative elementary size: RES (2D) and REV (3D)

**The question.** How large must an image be so that it represents the material? A piece that is too small gives different answers depending on where it was cut. The smallest size at which the answer no longer depends on the position is the **representative elementary volume** (REV; Bear, 1972), or representative elementary **size** (RES) for a 2D image.

**The method** (Amiri et al., 2024) uses $S_2$ as the property to test, because it contains the porosity, the specific surface and the feature size at once:

1. Choose a list of sub-image sizes, for example 32, 64, 128 pixels.
2. For each size, cut `n_samples` crops at random positions and compute $S_2$ and the scaled $F_2$ of every crop.
3. Average the curves of each size, and compute their standard deviation.
4. Compare with the curve of the whole image.

**How to read the result.** For a size below the REV, the mean curve differs from the curve of the whole image and the spread between crops is large. The REV is the smallest size whose mean curve lies on the whole-image curve with a small spread. Use 30 or more crops per size. The sizes must be smaller than the image.

The random positions are drawn with a fixed seed, so repeating the analysis gives the same result.

In [ ]:
def REV_RES(image, sizes, n_samples=30, seed=33):
    """S2 and F2 of random crops of several sizes, for a 2D image (RES) or a 3D volume (REV).

    Returns (s2_table, f2_table). Each table has the columns 'r', 'whole' (the whole image),
    and 'mean_<size>' and 'std_<size>' for every crop size.
    """
    np.random.seed(seed)
    s2_function = calculate_s2_periodic if image.ndim == 2 else calculate_s2_periodic_3d
    s2_columns, f2_columns = {}, {}

    for size in sizes:
        s2_crops, f2_crops = [], []
        for _ in range(n_samples):
            corner = [np.random.randint(0, n - size) for n in image.shape]      # random position of the crop
            window = tuple(slice(c, c + size) for c in corner)                  # size x size (x size) window
            s2 = s2_function(image[window])
            s2_crops.append(s2)
            f2_crops.append(cal_fn(s2, n=2))
        s2_crops, f2_crops = pd.DataFrame(s2_crops), pd.DataFrame(f2_crops)     # one row per crop, one column per r
        s2_columns[f'mean_{size}'], s2_columns[f'std_{size}'] = s2_crops.mean(), s2_crops.std()
        f2_columns[f'mean_{size}'], f2_columns[f'std_{size}'] = f2_crops.mean(), f2_crops.std()
        print(f'crops of size {size} done')

    print('whole image ...')
    s2_whole = s2_function(image)
    s2_columns = {'whole': pd.Series(s2_whole), **s2_columns}
    f2_columns = {'whole': pd.Series(cal_fn(s2_whole, n=2)), **f2_columns}

    s2_table = pd.DataFrame(s2_columns).rename_axis('r').reset_index()
    f2_table = pd.DataFrame(f2_columns).rename_axis('r').reset_index()
    return s2_table, f2_table


def plot_rev(s2_table, f2_table, save_as=None, ndim=2):
    """Mean curve of every crop size (line) with its standard deviation (band), and the whole image (black).
    ndim: 2 for RES on a 2D image (r in pixels), 3 for REV on a 3D volume (r in voxels)."""
    fig, axes = plt.subplots(1, 2, figsize=(2 * PANEL_W, PANEL_H))
    for ax, table, ylabel in zip(axes, (s2_table, f2_table), ('$S_2(r)$', '$F_2(r)$')):
        for column in table.columns:
            if column.startswith('mean_'):
                size = column.split('_')[1]
                mean, std = table[column], table[f'std_{size}']
                ax.plot(table['r'], mean, label=f'size {size}')
                ax.fill_between(table['r'], mean - std, mean + std, alpha=0.2)
        ax.plot(table['r'], table['whole'], color='black', lw=3, label='whole image')   # thicker than the other lines
        if table is s2_table:                              # S2 starts at phi, F2 at 1
            mark_phase_fraction(ax, table['whole'].iloc[0])
        ax.set_xlabel(r_label(ndim))
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
        ax.legend()
    _finish(fig, save_as)

## A10. Minkowski functionals

**What they are.** The Minkowski functionals are the basic geometric measures of a shape. In $d$ dimensions there are $d + 1$ of them:

| | 2D | 3D |
|---|---|---|
| size | area $A$ | volume $V$ |
| boundary | perimeter $P$ | surface area $S$ |
| curvature | | integral of the mean curvature $M$ |
| topology | Euler characteristic $\chi$ | Euler characteristic $\chi$ |

They matter because of Hadwiger's theorem: every measure of a shape that is additive, does not depend on position or orientation, and changes continuously with the shape, is a combination of these $d+1$ numbers (Mecke, 2000). They are widely used to characterise porous media (Armstrong et al., 2019).

**The Euler characteristic** $\chi$ counts connectivity. In 2D, $\chi$ = number of objects minus number of holes. In 3D, $\chi$ = objects minus tunnels (loops) plus enclosed cavities. Many isolated pores give a large positive $\chi$; a well-connected pore network with many loops gives a negative $\chi$.

**Densities.** Each functional is also divided by the size of the whole image (area in 2D, volume in 3D), which gives quantities that can be compared between images of different size: area fraction or porosity, specific perimeter or specific surface area, specific mean curvature, specific Euler characteristic.

**How they are computed.** With the QuantImPy library (Boelens & Tchelepi, 2021), which measures the functionals on the pixel grid (see also Legland et al., 2007). QuantImPy returns the functionals in a normalised convention; the constants in the code below convert them back to the plain geometric quantities (perimeter, surface area, ...). They were checked against shapes with known values (disk, sphere, cube), so leave them as they are.

`res` is the pixel size. With `res = 1` all results are in pixels.

In [ ]:
def minkowski_2d(image, res=1.0):
    """Minkowski functionals of a 2D binary image and their densities. res = pixel size."""
    if mk is None:
        raise ImportError('The Minkowski functionals need the quantimpy package.')
    area, length, euler = mk.functionals(image.astype(bool), (res, res))
    perimeter = length * (2 * np.pi)               # QuantImPy convention -> true perimeter
    euler = euler * np.pi                          # QuantImPy convention -> true Euler characteristic
    domain_area = image.size * res ** 2            # area of the whole image
    return {
        'area': area,
        'perimeter': perimeter,
        'euler': euler,
        'area_fraction': area / domain_area,              # no unit
        'specific_perimeter': perimeter / domain_area,    # 1 / length
        'specific_euler': euler / domain_area,            # 1 / length^2
    }


def minkowski_3d(image, res=1.0):
    """Minkowski functionals of a 3D binary volume and their densities. res = voxel size."""
    if mk is None:
        raise ImportError('The Minkowski functionals need the quantimpy package.')
    volume, surface, curvature, euler = mk.functionals(image.astype(bool), (res, res, res))
    surface_area = surface * 8.0                   # QuantImPy convention -> true surface area
    mean_curv = curvature * (2 * np.pi ** 2)       # -> integral of the mean curvature
    euler = euler * (4 * np.pi / 3)                # -> true Euler characteristic
    domain_volume = image.size * res ** 3          # volume of the whole image
    return {
        'volume': volume,
        'surface_area': surface_area,
        'mean_curv': mean_curv,
        'euler': euler,
        'porosity': volume / domain_volume,                       # no unit
        'specific_surface_area': surface_area / domain_volume,    # 1 / length
        'specific_mean_curv': mean_curv / domain_volume,          # 1 / length^2
        'specific_euler': euler / domain_volume,                  # 1 / length^3
    }

## A11. Connected components and their measurements

**Labelling.** A connected component is a group of phase pixels that touch each other: one pore, one grain, one crack. Labelling gives every component its own number, so that each one can be measured separately. It is done with scikit-image (van der Walt et al., 2014).

**Connectivity** decides which pixels count as touching:

| | faces only | faces + edges | faces + edges + corners |
|---|---|---|---|
| 2D | 4 | | 8 |
| 3D | 6 | 18 | 26 |

With a low connectivity, pixels that only touch diagonally belong to different components, so you get more and smaller components. Unlike in $C_2$ (A3), the image is **not** periodic here: a component stops at the image border.

**Measurements for every component.** `res` is the pixel size, `min_size` removes components smaller than that number of pixels (segmentation noise) before measuring.

| Column | Meaning | |
|---|---|---|
| `pixel_count` / `voxel_count` | number of pixels | 2D / 3D |
| `area` / `volume` | count $\times$ pixel area or voxel volume | 2D / 3D |
| `equivalent_diameter` | diameter of the circle (sphere) with the same area (volume) | 2D, 3D |
| `aspect_ratio` | longest / shortest axis of the ellipse (ellipsoid) with the same second moments; 1 = round | 2D, 3D |
| `perimeter` | length of the outline | 2D |
| `specific_perimeter` | perimeter / area of the component | 2D |
| `circularity` | $4\pi A / P^2$; 1 for a circle, lower for irregular shapes | 2D |
| `surface_area` | area of a triangle mesh fitted around the component (marching cubes; Lorensen & Cline, 1987) | 3D, advanced |
| `specific_surface_area` | surface area / volume of the component | 3D, advanced |
| `sphericity` | $\pi^{1/3}(6V)^{2/3} / S$; 1 for a sphere (Wadell, 1935) | 3D, advanced |
| `elongation` | intermediate / longest axis; small = elongated (rod) | 3D, advanced |
| `flatness` | shortest / intermediate axis; small = flat (plate) | 3D, advanced |
| `inclination_deg` | angle between the longest axis and Z; 0 = along Z, 90 = in the slice plane | 3D, advanced |
| `azimuth_deg` | direction of the longest axis within the slice plane, from the X axis (0 to 180) | 3D, advanced |

The advanced 3D measurements need a surface mesh or the principal axes of every component, which takes longer when there are thousands of components. They are unreliable for components of only a few voxels: use `min_size` to leave those out.

**The principal axes.** The inertia tensor of a component describes how its voxels are spread around its centre. Its eigenvectors are the three principal axes of the shape (long, intermediate, short), and its eigenvalues give their lengths. The longest axis belongs to the smallest eigenvalue. Its direction is reported as the two angles `inclination_deg` and `azimuth_deg`.

In [ ]:
CONNECTIVITY_RANK = {4: 1, 8: 2,             # 2D: number of neighbours -> scikit-image "connectivity"
                     6: 1, 18: 2, 26: 3}     # 3D


def connected_components(image, connectivity):
    """Label the connected components of a 2D image (connectivity 4 or 8) or a 3D volume (6, 18 or 26).

    Returns a dictionary:
        'labels'         : array like the image, 0 = background, 1 .. N = component number
        'num_components' : N
        'table'          : one row per component (label and pixel/voxel count), largest first
    """
    allowed = (4, 8) if image.ndim == 2 else (6, 18, 26)
    if connectivity not in allowed:
        raise ValueError(f'connectivity must be one of {allowed} for a {image.ndim}D image')
    labels, num_components = label(image.astype(bool), connectivity=CONNECTIVITY_RANK[connectivity], return_num=True)

    count_column = 'pixel_count' if image.ndim == 2 else 'voxel_count'
    table = pd.DataFrame({'label': np.arange(1, num_components + 1),
                          count_column: np.bincount(labels.ravel())[1:]})     # size of every label
    table = table.sort_values(count_column, ascending=False).reset_index(drop=True)
    return {'labels': labels, 'num_components': num_components, 'table': table}


def connected_components_2d_stack(stack, connectivity=8):
    """Label every frame of a 2D time series (T, Y, X) separately.

    Labelling the stack as a 3D volume would connect components across time. Here each frame is
    labelled on its own, and the label numbers continue from frame to frame so that each is unique.
    The table has an extra column 'frame'.
    """
    labels = np.zeros(stack.shape, dtype=np.int32)
    tables = []
    offset = 0
    for t in range(stack.shape[0]):
        frame = connected_components(stack[t], connectivity)
        labels[t] = np.where(frame['labels'] > 0, frame['labels'] + offset, 0)
        table = frame['table'].copy()
        table['label'] += offset
        table.insert(1, 'frame', t)
        tables.append(table)
        offset += frame['num_components']
    return {'labels': labels, 'num_components': offset, 'table': pd.concat(tables, ignore_index=True)}


def _safe_prop(region, name):
    """Read one property of a component. Some fail for degenerate shapes (one pixel thick): return NaN then."""
    try:
        return getattr(region, name)
    except ValueError:
        return np.nan


def compute_shape_measurements(labels, res=1.0, only_labels=None):
    """Basic shape measurements of every component (2D or 3D). Returns a table with one row per component."""
    is_3d = labels.ndim == 3
    rows = []
    for region in regionprops(labels):
        if only_labels is not None and region.label not in only_labels:
            continue
        major = _safe_prop(region, 'axis_major_length') * res
        minor = _safe_prop(region, 'axis_minor_length') * res
        row = {'label': region.label,
               'equivalent_diameter': _safe_prop(region, 'equivalent_diameter_area') * res,
               'aspect_ratio': major / minor if minor > 0 else np.nan}
        if not is_3d:
            perimeter = _safe_prop(region, 'perimeter') * res
            area = region.area * res ** 2
            row['perimeter'] = perimeter
            row['specific_perimeter'] = perimeter / area if area > 0 else np.nan
            row['circularity'] = 4 * np.pi * area / perimeter ** 2 if perimeter > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame({'label': []})


def _mesh_area_and_volume(region):
    """Surface area and enclosed volume of the marching-cubes mesh around one component (in voxels)."""
    try:
        padded = np.pad(region.image, 1)                       # one empty voxel around, so the surface closes
        verts, faces, _, _ = marching_cubes(padded, level=0.5)
        area = mesh_surface_area(verts, faces)
        # volume inside a closed mesh: sum of the signed volumes of the tetrahedra (triangle + origin)
        v0, v1, v2 = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
        volume = abs(np.sum(np.einsum('ij,ij->i', v0, np.cross(v1, v2))) / 6.0)
        return area, volume
    except (RuntimeError, ValueError):
        return np.nan, np.nan


def _axes_and_orientation(region):
    """Lengths of the three principal axes of one component, and the direction of the longest one."""
    try:
        eigenvalues, eigenvectors = np.linalg.eigh(region.inertia_tensor)       # eigenvalues in ascending order
    except np.linalg.LinAlgError:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    # axis lengths of the ellipsoid with the same inertia tensor (smallest eigenvalue -> longest axis)
    lengths = np.sqrt(np.clip(10 * eigenvalues.sum() - 20 * eigenvalues, 0, None))
    major, intermediate, minor = lengths

    v0, v1, v2 = eigenvectors[:, 0]                # direction of the longest axis, as (Z, Y, X) components
    if v0 < 0:                                     # an axis has no head or tail: make it point "up"
        v0, v1, v2 = -v0, -v1, -v2
    inclination = np.degrees(np.arccos(np.clip(v0, -1, 1)))     # angle from Z: 0 = along Z, 90 = in-plane
    azimuth = np.degrees(np.arctan2(v1, v2)) % 180.0            # in-plane direction from X, folded to 0..180
    if azimuth >= 180.0:
        azimuth = 0.0
    return major, intermediate, minor, inclination, azimuth


def compute_advanced_shape_measurements(labels, res=1.0, only_labels=None):
    """Advanced 3D measurements: surface area, specific surface area, sphericity, elongation,
    flatness, inclination and azimuth of every component."""
    rows = []
    for region in regionprops(labels):
        if only_labels is not None and region.label not in only_labels:
            continue
        mesh_area, mesh_volume = _mesh_area_and_volume(region)
        surface_area = mesh_area * res ** 2
        volume = region.area * res ** 3                        # volume from the voxel count
        # the sphericity compares a surface with a volume: both must come from the same mesh,
        # otherwise small components can score above 1
        mesh_volume = mesh_volume * res ** 3
        major, intermediate, minor, inclination, azimuth = _axes_and_orientation(region)
        rows.append({
            'label': region.label,
            'surface_area': surface_area,
            'specific_surface_area': surface_area / volume if volume > 0 else np.nan,
            'sphericity': np.pi ** (1 / 3) * (6 * mesh_volume) ** (2 / 3) / surface_area if surface_area > 0 else np.nan,
            'elongation': intermediate / major if major > 0 else np.nan,
            'flatness': minor / intermediate if intermediate > 0 else np.nan,
            'inclination_deg': inclination,
            'azimuth_deg': azimuth,
        })
    return pd.DataFrame(rows) if rows else pd.DataFrame({'label': []})


def measure_components(cc, res=1.0, min_size=1, advanced=False):
    """Measure the components found by connected_components().

    cc: the dictionary returned by connected_components()
    res: pixel size.  min_size: smaller components (in pixels) are left out.
    advanced: also compute the slower 3D measurements (surface area, sphericity, orientation, ...).
    """
    is_3d = cc['labels'].ndim == 3
    count_column, size_column, power = ('voxel_count', 'volume', 3) if is_3d else ('pixel_count', 'area', 2)

    table = cc['table'].copy()
    table[size_column] = table[count_column] * res ** power               # physical area or volume
    table = table[table[count_column] >= min_size].reset_index(drop=True)
    kept = set(table['label'])

    table = table.merge(compute_shape_measurements(cc['labels'], res, kept), on='label', how='left')
    if is_3d and advanced:
        table = table.merge(compute_advanced_shape_measurements(cc['labels'], res, kept), on='label', how='left')
    return table


def show_labels(labels, title='', save_as=None):
    """Show a 2D label image with a different colour for every component (background black)."""
    colors = np.random.default_rng(0).random((int(labels.max()) + 1, 3))
    colors[0] = 0
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.imshow(colors[labels], interpolation='nearest')
    ax.set_title(title)
    ax.axis('off')
    _finish(fig, save_as)


def plot_histogram(values, xlabel, percentile=99.0, bins=30, log_y=False, save_as=None):
    """Histogram of one measurement. Values above the given percentile are grouped into the last bin,
    so that a few very large components do not squeeze all the others into the first bin."""
    values = np.asarray(values, dtype=float)
    cutoff = np.percentile(values, percentile)
    if cutoff <= values.min():
        cutoff = values.max()
    fig, ax = plt.subplots(figsize=(PANEL_W, PANEL_H))
    ax.hist(np.clip(values, None, cutoff), bins=np.linspace(values.min(), cutoff, bins + 1))
    if log_y:
        ax.set_yscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('number of components')
    ax.set_title(f'n = {len(values)}')
    ax.grid(alpha=0.3)
    _finish(fig, save_as)

## A12. Helpers for image series (time steps or slices)

For a series, the single-image analyses above are simply repeated on every frame. `analyse_stack` does this in **one pass**: each frame is taken, analysed with everything you switched on, and released again. Only one frame is in memory at a time, which is what makes long 4D series possible.

`frames` can be:
- an array `(T, Y, X)`: looping over it gives one 2D image per step (Part 3, and the Z slices in Part 2);
- `frames_from_files(paths)`: reads one file at a time from disk (Part 4, where the whole series may not fit in memory).

The plot functions below show the result against the step number.

In [ ]:
def frames_from_files(paths, foreground_value=1):
    """Give one binarized image or volume at a time, read from disk only when it is needed."""
    for path in paths:
        yield binarize(load_image(path), foreground_value)


def analyse_stack(frames, smds=('s2',), chord_length=True, minkowski=True,
                  pixel_size=1.0, num_points=2, nbins=40):
    """Run the single-image analyses on every frame of a series (2D images or 3D volumes).

    smds: descriptors to compute on every frame (empty list = none). 3D frames: 's2', 'c2', 'L' only.
    Returns a dictionary of lists with one entry per frame:
        'raw', 'scaled'  : the two tables of compute_2d/3d_polytope_curves
        'chord'          : the dictionary of compute_chord_length_result
        'minkowski'      : the dictionary of minkowski_2d / minkowski_3d
    """
    results = {'raw': [], 'scaled': [], 'chord': [], 'minkowski': []}
    for frame in tqdm(frames):
        is_3d = frame.ndim == 3
        if smds:
            compute_curves = compute_3d_polytope_curves if is_3d else compute_2d_polytope_curves
            raw, scaled = compute_curves(frame, smds)
            results['raw'].append(raw)
            results['scaled'].append(scaled)
        if chord_length:
            results['chord'].append(compute_chord_length_result(frame, num_points=num_points, nbins=nbins))
        if minkowski:
            compute_minkowski = minkowski_3d if is_3d else minkowski_2d
            results['minkowski'].append(compute_minkowski(frame, res=pixel_size))
    return results


def stack_tables(tables, steps):
    """Join the tables of all steps into one long table with a column 'step' (handy for saving)."""
    return pd.concat([table.assign(step=step) for table, step in zip(tables, steps)], ignore_index=True)


def plot_curve_evolution(results, steps, name, step_label='time step', save_as=None, ndim=2):
    """Raw (left) and scaled (right) curves of one function, one line per step, coloured by step.
    ndim: dimensionality of ONE step - 2 for 2D images or slices (pixels), 3 for 3D volumes (voxels)."""
    fig, axes = plt.subplots(1, 2, figsize=(2 * PANEL_W, PANEL_H), constrained_layout=True)
    cmap, norm = plt.cm.viridis, plt.Normalize(min(steps), max(steps))
    skip = 2 if name in ('p3h', 'p3v') else 1
    for ax, key, ylabel in zip(axes, ('raw', 'scaled'), ('raw', 'scaled')):
        for step, table in zip(steps, results[key]):
            ax.plot(table['r'][::skip], table[name][::skip], color=cmap(norm(step)), lw=1.2, alpha=0.85)
        ax.set_title(f'{name.upper()} ({ylabel})')
        ax.set_xlabel(r_label(ndim))
        ax.set_ylabel('$P_n(r)$' if key == 'raw' else '$f_n(r)$')
        ax.grid(alpha=0.3)
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, label=step_label, shrink=0.8)
    _finish(fig, save_as)


def plot_omega(metrics, step_label='time step', save_as=None):
    """Omega (left) and Delta-Omega (right) against the step, one line per function."""
    fig, axes = plt.subplots(1, 2, figsize=(2 * PANEL_W, PANEL_H), constrained_layout=True)
    for ax, prefix, ylabel, title in zip(axes, ('omega_', 'delta_omega_'), ('$\\Omega$', '$\\Delta\\Omega$'),
                                         ('$\\Omega$  (difference from the reference step)',
                                          '$\\Delta\\Omega$  (difference from the previous step)')):
        for column in metrics.columns:
            if column.startswith(prefix):
                ax.plot(metrics['step'], metrics[column], marker='o', ms=3, label=column[len(prefix):].upper())
        ax.set_title(title)
        ax.set_xlabel(step_label)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
        ax.legend()
    _finish(fig, save_as)


def plot_chord_evolution(chord_results, steps, show=None, max_length=100, nbins=40,
                         step_label='time step', save_as=None, unit='pixels'):
    """Mean chord length against the step (left), and the chord-length distribution of a few steps (right).

    show: positions in the series whose distribution is drawn (default: first, middle, last).
    max_length: longest chord shown in the distributions.
    unit: 'pixels' for 2D images, 'voxels' for 3D volumes.
    """
    if show is None:
        show = sorted({0, len(steps) // 2, len(steps) - 1})
    fig, (ax_mean, ax_pdf) = plt.subplots(1, 2, figsize=(2 * PANEL_W, PANEL_H), constrained_layout=True)

    ax_mean.plot(steps, [result['ell_via_s2'] for result in chord_results], marker='o', label='via S2')
    ax_mean.plot(steps, [result['ell_direct'] for result in chord_results], marker='s', label='direct')
    ax_mean.set_xlabel(step_label)
    ax_mean.set_ylabel(f'mean chord length ({unit})')
    ax_mean.grid(alpha=0.3)
    ax_mean.legend()

    for i in show:                                   # same bins for every step, so the curves can be compared
        p_z, edges = np.histogram(chord_results[i]['chords'], bins=nbins, range=(0, max_length), density=True)
        ax_pdf.stairs(p_z, edges, label=f'{step_label} {steps[i]}', lw=1.5)
    ax_pdf.set_xlabel(f'chord length ({unit})')
    ax_pdf.set_ylabel('$p(z)$')
    ax_pdf.grid(alpha=0.3)
    ax_pdf.legend()
    _finish(fig, save_as)


def plot_columns(table, x='step', xlabel=None, save_as=None):
    """One small panel for every column of a table, each plotted against the column `x`."""
    columns = [column for column in table.columns if column != x]
    n_cols = min(4, len(columns))
    n_rows = math.ceil(len(columns) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(PANEL_W * n_cols, PANEL_H * n_rows), squeeze=False, constrained_layout=True)
    for ax, column in zip(axes.ravel(), columns):
        ax.plot(table[x], table[column], marker='o', ms=3)
        ax.set_title(column)
        ax.set_xlabel(xlabel or x)
        ax.set_ylabel(column)               # the saved file has no titles
        ax.grid(alpha=0.3)
    for ax in axes.ravel()[len(columns):]:
        ax.axis('off')
    _finish(fig, save_as)

# Part 1. 2D image

Set the path to your image and the three values below, then run the cells. Each analysis (1.1 to 1.5) is independent, so you can run only the ones you need.

In [ ]:
PATH_2D = 'demo_data/image_2d.tif'      # <-- path to your 2D image (.tif)
FOREGROUND_2D = 1                       # pixel value of the phase to analyse (for example the pores)
PIXEL_SIZE_2D = 1.0                     # size of one pixel, for example 11.4 (micrometres). 1.0 = results in pixels
OUT_2D = os.path.join(OUTPUT_DIR, '2D') # where the results of this part are saved

img2d = load_image(PATH_2D)
describe(img2d, 'loaded image')
img2d = binarize(img2d, FOREGROUND_2D)

# The image does not need to be square: every function runs up to r = min(img2d.shape) // 2 - 1.

print(f'phase fraction (porosity): {img2d.mean():.4f}')
show_image(img2d, 'binary image (white = phase 1)')

## 1.1 Correlation functions ($S_2$, $C_2$, $L$, polytopes)

Choose the functions in `SMDS_2D`. The raw and scaled curves are returned as two tables, plotted, and saved. Methods: A2 to A6.

In [ ]:
SMDS_2D = ['s2', 'c2', 'L', 'p3h', 'p3v', 'p4', 'p6']     # remove the ones you do not need

raw_2d, scaled_2d = compute_2d_polytope_curves(img2d, SMDS_2D)

plot_curves(raw_2d, scaled_2d, title='2D image', save_as=os.path.join(OUT_2D, 'smds.png'))
save_table(raw_2d, os.path.join(OUT_2D, 'smds_raw.csv'))
save_table(scaled_2d, os.path.join(OUT_2D, 'smds_scaled.csv'))
raw_2d.head()

## 1.2 Chord length

Mean chord length from the $S_2$ slope and from direct measurement, and the chord-length distribution. Method: A8.

In [ ]:
chord_2d = compute_chord_length_result(img2d, num_points=2, nbins=40)

print(f"mean chord length via S2 : {chord_2d['ell_via_s2']:.3f} pixels")
print(f"mean chord length direct : {chord_2d['ell_direct']:.3f} pixels   ({len(chord_2d['chords'])} chords)")

plot_chord(chord_2d, unit='pixels', save_as=os.path.join(OUT_2D, 'chord_length.png'))
save_table(pd.DataFrame({'z': chord_2d['z'], 'p_z': chord_2d['p_z']}), os.path.join(OUT_2D, 'chord_length_distribution.csv'))

## 1.3 Representative elementary size (RES)

$S_2$ and $F_2$ of random crops of several sizes, compared with the whole image. Method: A9. The sizes must be smaller than the image.

In [ ]:
SIZES_2D = [32, 64, 128]      # crop sizes in pixels
N_SAMPLES_2D = 30             # number of random crops per size

res_s2_2d, res_f2_2d = REV_RES(img2d, SIZES_2D, N_SAMPLES_2D)

plot_rev(res_s2_2d, res_f2_2d, save_as=os.path.join(OUT_2D, 'res.png'))
save_table(res_s2_2d, os.path.join(OUT_2D, 'res_s2.csv'))
save_table(res_f2_2d, os.path.join(OUT_2D, 'res_f2.csv'))

## 1.4 Minkowski functionals

Area, perimeter and Euler characteristic of the phase, and their densities. Method: A10. With `PIXEL_SIZE_2D = 1` the values are in pixels.

In [ ]:
mink_2d = pd.DataFrame([minkowski_2d(img2d, res=PIXEL_SIZE_2D)])     # a table with a single row

save_table(mink_2d, os.path.join(OUT_2D, 'minkowski.csv'))
mink_2d.T.rename(columns={0: 'value'})                               # shown as a column, easier to read

## 1.5 Connected components and measurements

Label the separate objects of the phase, then measure each one. Method: A11.

In [ ]:
CONNECTIVITY_2D = 8           # 4 (pixels touching by an edge) or 8 (edge or corner)
MIN_SIZE_2D = 1               # leave out components smaller than this number of pixels

cc_2d = connected_components(img2d, CONNECTIVITY_2D)
print(cc_2d['num_components'], 'components found')

table_2d = measure_components(cc_2d, res=PIXEL_SIZE_2D, min_size=MIN_SIZE_2D)
print(len(table_2d), 'components kept after the size filter')

show_labels(cc_2d['labels'], 'connected components', save_as=os.path.join(OUT_2D, 'components.png'))
save_table(table_2d, os.path.join(OUT_2D, 'components.csv'))
table_2d.head(10)

Histogram of one measurement (any column of the table above):

In [ ]:
COLUMN_2D = 'equivalent_diameter'

plot_histogram(table_2d[COLUMN_2D].dropna(), COLUMN_2D, percentile=99, bins=30, log_y=False,
               save_as=os.path.join(OUT_2D, f'histogram_{COLUMN_2D}.png'))

Optional: keep only the components that passed the size filter and continue with that cleaned image. Uncomment the last line to use it for the analyses above (then run them again).

In [ ]:
img2d_cleaned = np.isin(cc_2d['labels'], table_2d['label']).astype(np.uint8)
print(f'phase fraction before: {img2d.mean():.4f}   after cleaning: {img2d_cleaned.mean():.4f}')
# img2d = img2d_cleaned

# Part 2. 3D volume

`PATH_3D` can be either a multi-page `.tif` file, or a **folder** that contains the 2D slices as separate `.tif` files (they are sorted by the number in their file name and stacked).

Large volumes take time: for 512³ voxels expect about a minute for $S_2$ and several minutes (and a few GB of memory) for $C_2$. To try things out quickly, work on a crop first (see the comment in the cell).

In [ ]:
PATH_3D = 'demo_data/volume_3d.tif'     # <-- a multi-page .tif file, or a folder of 2D slices
FOREGROUND_3D = 1                       # voxel value of the phase to analyse
PIXEL_SIZE_3D = 1.0                     # size of one voxel. 1.0 = results in voxels
OUT_3D = os.path.join(OUTPUT_DIR, '3D')

if os.path.isdir(PATH_3D):
    slice_files, slice_numbers = list_files_sorted(PATH_3D, number_position=-1)
    vol = load_stack(slice_files)
else:
    vol = load_image(PATH_3D)
describe(vol, 'loaded volume')
vol = binarize(vol, FOREGROUND_3D)

# To work on a smaller piece first:
# vol = vol[:200, :200, :200]

print(f'phase fraction (porosity): {vol.mean():.4f}')
show_image(vol, 'binary volume')

## 2.1 Correlation functions ($S_2$, $C_2$, $L$)

In 3D the available functions are `'s2'`, `'c2'` and `'L'`. Methods: A2 to A4.

In [ ]:
SMDS_3D = ['s2', 'c2', 'L']

raw_3d, scaled_3d = compute_3d_polytope_curves(vol, SMDS_3D)

plot_curves(raw_3d, scaled_3d, title='3D volume', save_as=os.path.join(OUT_3D, 'smds.png'), ndim=3)
save_table(raw_3d, os.path.join(OUT_3D, 'smds_raw.csv'))
save_table(scaled_3d, os.path.join(OUT_3D, 'smds_scaled.csv'))
raw_3d.head()

## 2.2 Chord length

In [ ]:
chord_3d = compute_chord_length_result(vol, num_points=2, nbins=40)

print(f"mean chord length via S2 : {chord_3d['ell_via_s2']:.3f} voxels")
print(f"mean chord length direct : {chord_3d['ell_direct']:.3f} voxels   ({len(chord_3d['chords'])} chords)")

plot_chord(chord_3d, unit='voxels', save_as=os.path.join(OUT_3D, 'chord_length.png'))
save_table(pd.DataFrame({'z': chord_3d['z'], 'p_z': chord_3d['p_z']}), os.path.join(OUT_3D, 'chord_length_distribution.csv'))

## 2.3 Representative elementary volume (REV)

The crops are cubes of the given sizes, taken at random positions. Method: A9.

In [ ]:
SIZES_3D = [16, 32, 64]       # edge length of the cubic crops, in voxels (smaller than the volume)
N_SAMPLES_3D = 30             # number of random crops per size

rev_s2_3d, rev_f2_3d = REV_RES(vol, SIZES_3D, N_SAMPLES_3D)

plot_rev(rev_s2_3d, rev_f2_3d, save_as=os.path.join(OUT_3D, 'rev.png'), ndim=3)
save_table(rev_s2_3d, os.path.join(OUT_3D, 'rev_s2.csv'))
save_table(rev_f2_3d, os.path.join(OUT_3D, 'rev_f2.csv'))

## 2.4 Minkowski functionals

Volume, surface area, integral mean curvature and Euler characteristic, and their densities. Method: A10.

In [ ]:
mink_3d = pd.DataFrame([minkowski_3d(vol, res=PIXEL_SIZE_3D)])

save_table(mink_3d, os.path.join(OUT_3D, 'minkowski.csv'))
mink_3d.T.rename(columns={0: 'value'})

## 2.5 Connected components and measurements

Set `ADVANCED_3D = True` to add surface area, sphericity, elongation, flatness and orientation. These are slower when there are thousands of components. Method: A11.

In [ ]:
CONNECTIVITY_3D = 26          # 6 (faces), 18 (faces + edges) or 26 (faces + edges + corners)
MIN_SIZE_3D = 10              # leave out components smaller than this number of voxels
ADVANCED_3D = True            # also compute the mesh-based and orientation measurements

cc_3d = connected_components(vol, CONNECTIVITY_3D)
print(cc_3d['num_components'], 'components found')

table_3d = measure_components(cc_3d, res=PIXEL_SIZE_3D, min_size=MIN_SIZE_3D, advanced=ADVANCED_3D)
print(len(table_3d), 'components kept after the size filter')

z_middle = vol.shape[0] // 2
show_labels(cc_3d['labels'][z_middle], f'connected components, slice z = {z_middle}',
            save_as=os.path.join(OUT_3D, 'components_slice.png'))
save_table(table_3d, os.path.join(OUT_3D, 'components.csv'))
table_3d.head(10)

In [ ]:
COLUMN_3D = 'equivalent_diameter'

plot_histogram(table_3d[COLUMN_3D].dropna(), COLUMN_3D, percentile=99, bins=30, log_y=False,
               save_as=os.path.join(OUT_3D, f'histogram_{COLUMN_3D}.png'))

## 2.6 Slice by slice along Z

A volume can also be analysed as a series of 2D images: the 2D descriptors (including the polytope functions) are computed on every Z slice, and $\Omega$ / $\Delta\Omega$ show how the structure changes from slice to slice. This reveals layering or gradients along Z. Methods: A6, A7, A12.

`Z_STEP = 5` uses every fifth slice. `Z_REFERENCE` is the position, in the list of used slices, of the slice that $\Omega$ compares with.

In [ ]:
SMDS_Z = ['s2', 'c2']         # 2D functions to compute on every slice
Z_STEP = 4                    # use every Z_STEP-th slice (1 = all slices)
Z_REFERENCE = 0               # reference slice for Omega: position in the list of used slices

z_positions = list(range(0, vol.shape[0], Z_STEP))
z_results = analyse_stack(vol[::Z_STEP], smds=SMDS_Z, chord_length=False, minkowski=False)
z_metrics = evolution_metrics(z_results['raw'], z_positions, reference_index=Z_REFERENCE)

for name in SMDS_Z:
    plot_curve_evolution(z_results, z_positions, name, step_label='slice (z)',
                         save_as=os.path.join(OUT_3D, f'slices_{name}.png'))
plot_omega(z_metrics, step_label='slice (z)', save_as=os.path.join(OUT_3D, 'slices_omega.png'))

save_table(stack_tables(z_results['raw'], z_positions), os.path.join(OUT_3D, 'slices_smds_raw.csv'))
save_table(stack_tables(z_results['scaled'], z_positions), os.path.join(OUT_3D, 'slices_smds_scaled.csv'))
save_table(z_metrics, os.path.join(OUT_3D, 'slices_omega.csv'))

# Part 3. 2D + time

One folder with one 2D `.tif` image per time step. The files are sorted by a number in their name, and that number is used as the time-step label in the plots and tables.

- `NUMBER_POSITION_2DT`: which number of the file name is the time step (`-1` = the last number, `0` = the first, `1` = the second, ...).
- `TIME_STEP_2DT`: use every n-th file only.

The whole series is loaded into memory as one array `(T, Y, X)`, which is fine for 2D images.

In [ ]:
FOLDER_2DT = 'demo_data/time_series_2d'   # <-- folder with one 2D .tif per time step
NUMBER_POSITION_2DT = -1                  # which number in the file name is the time step
TIME_STEP_2DT = 1                         # 1 = every file, 2 = every second file, ...
FOREGROUND_2DT = 1                        # pixel value of the phase to analyse
PIXEL_SIZE_2DT = 1.0                      # size of one pixel. 1.0 = results in pixels
OUT_2DT = os.path.join(OUTPUT_DIR, '2D_time')

files_2dt, steps_2dt = list_files_sorted(FOLDER_2DT, NUMBER_POSITION_2DT)
files_2dt, steps_2dt = files_2dt[::TIME_STEP_2DT], steps_2dt[::TIME_STEP_2DT]
print(len(files_2dt), 'time steps:', steps_2dt)

stack_2dt = binarize(load_stack(files_2dt), FOREGROUND_2DT)
print('stack shape (T, Y, X):', stack_2dt.shape)
show_image(stack_2dt[0], f'first time step ({steps_2dt[0]})')
show_image(stack_2dt[-1], f'last time step ({steps_2dt[-1]})')

## 3.1 Run the analyses on every time step

One pass over the series computes everything that is switched on. Switch off what you do not need: set `SMDS_2DT = []`, or `chord_length=False`, or `minkowski=False`.

In [ ]:
SMDS_2DT = ['s2', 'c2', 'L', 'p4']       # any of 's2', 'c2', 'L', 'p3h', 'p3v', 'p4', 'p6'

results_2dt = analyse_stack(stack_2dt, smds=SMDS_2DT, chord_length=True, minkowski=True,
                            pixel_size=PIXEL_SIZE_2DT)

## 3.2 Evolution of the correlation functions, $\Omega$ and $\Delta\Omega$

One figure per function, with one curve per time step (dark = early, bright = late), then the two evolution metrics. Method: A7.

`REFERENCE_2DT` is the position in the series of the step that $\Omega$ compares with (0 = the first step).

In [ ]:
REFERENCE_2DT = 0             # reference step for Omega: position in the series (0 = first)
SIGNED_2DT = False            # True: Delta-Omega keeps its sign (net change)

for name in SMDS_2DT:
    plot_curve_evolution(results_2dt, steps_2dt, name, save_as=os.path.join(OUT_2DT, f'evolution_{name}.png'))

metrics_2dt = evolution_metrics(results_2dt['raw'], steps_2dt, reference_index=REFERENCE_2DT, signed=SIGNED_2DT)
plot_omega(metrics_2dt, save_as=os.path.join(OUT_2DT, 'omega.png'))

save_table(stack_tables(results_2dt['raw'], steps_2dt), os.path.join(OUT_2DT, 'smds_raw.csv'))
save_table(stack_tables(results_2dt['scaled'], steps_2dt), os.path.join(OUT_2DT, 'smds_scaled.csv'))
save_table(metrics_2dt, os.path.join(OUT_2DT, 'omega.csv'))
metrics_2dt

## 3.3 Evolution of the chord length

Left: mean chord length (both methods) against time. Right: chord-length distribution of the first, middle and last step. To compare other steps, pass their positions, for example `show=[0, 3, 7]`.

In [ ]:
plot_chord_evolution(results_2dt['chord'], steps_2dt, show=None, max_length=100,
                     save_as=os.path.join(OUT_2DT, 'chord_length.png'))

chord_table_2dt = pd.DataFrame({'step': steps_2dt,
                                'ell_via_s2': [result['ell_via_s2'] for result in results_2dt['chord']],
                                'ell_direct': [result['ell_direct'] for result in results_2dt['chord']]})
save_table(chord_table_2dt, os.path.join(OUT_2DT, 'chord_length.csv'))
chord_table_2dt

## 3.4 Evolution of the Minkowski functionals

In [ ]:
mink_table_2dt = pd.DataFrame(results_2dt['minkowski'])
mink_table_2dt.insert(0, 'step', steps_2dt)

plot_columns(mink_table_2dt, x='step', xlabel='time step', save_as=os.path.join(OUT_2DT, 'minkowski.png'))
save_table(mink_table_2dt, os.path.join(OUT_2DT, 'minkowski.csv'))
mink_table_2dt

## 3.5 Connected components in every frame

Every frame is labelled on its own, so components are **not** tracked through time: the same pore has a different label in the next frame. The table lists every component with the frame it belongs to; the plot shows how the number of components and their mean size change.

In [ ]:
CONNECTIVITY_2DT = 8          # 4 or 8

cc_2dt = connected_components_2d_stack(stack_2dt, CONNECTIVITY_2DT)
cc_table_2dt = cc_2dt['table']
cc_table_2dt.insert(1, 'step', np.array(steps_2dt)[cc_table_2dt['frame']])     # time-step label of every component
print(cc_2dt['num_components'], 'components in', len(steps_2dt), 'frames')

per_step_2dt = cc_table_2dt.groupby('step')['pixel_count'].agg(number_of_components='count', mean_size='mean').reset_index()
plot_columns(per_step_2dt, x='step', xlabel='time step', save_as=os.path.join(OUT_2DT, 'components_per_step.png'))

show_labels(cc_2dt['labels'][0], f'components, time step {steps_2dt[0]}')
show_labels(cc_2dt['labels'][-1], f'components, time step {steps_2dt[-1]}')
save_table(cc_table_2dt, os.path.join(OUT_2DT, 'components.csv'))

# Part 4. 4D: 3D + time

One folder with one **multi-page `.tif` volume per time step**. A 4D data set is often too large to load at once, so here the volumes are never stacked: each one is read, analysed and released before the next one is read. Memory use stays at one volume, however many time steps there are.

Because reading and analysing a large volume takes time, everything is computed in a single pass (4.1). Choose there what you need. For large volumes, start with a few time steps (`TIME_STEP_4D`) to see how long one volume takes.

In [ ]:
FOLDER_4D = 'demo_data/time_series_3d'    # <-- folder with one multi-page .tif (3D volume) per time step
NUMBER_POSITION_4D = -1                   # which number in the file name is the time step
TIME_STEP_4D = 1                          # 1 = every file, 2 = every second file, ...
FOREGROUND_4D = 1                         # voxel value of the phase to analyse
PIXEL_SIZE_4D = 1.0                       # size of one voxel. 1.0 = results in voxels
OUT_4D = os.path.join(OUTPUT_DIR, '4D')

files_4d, steps_4d = list_files_sorted(FOLDER_4D, NUMBER_POSITION_4D)
files_4d, steps_4d = files_4d[::TIME_STEP_4D], steps_4d[::TIME_STEP_4D]
print(len(files_4d), 'time steps:', steps_4d)

first_volume = load_image(files_4d[0])     # only the first volume is loaded here, as a preview
describe(first_volume, 'first volume')
show_image(binarize(first_volume, FOREGROUND_4D), f'time step {steps_4d[0]}')

## 4.1 Run the analyses on every time step

`frames_from_files` hands the volumes to `analyse_stack` one at a time. In 3D the available functions are `'s2'`, `'c2'` and `'L'`. $C_2$ is the slowest one by far, so leave it out for a first run on large volumes.

In [ ]:
SMDS_4D = ['s2', 'c2', 'L']

results_4d = analyse_stack(frames_from_files(files_4d, FOREGROUND_4D), smds=SMDS_4D,
                           chord_length=True, minkowski=True, pixel_size=PIXEL_SIZE_4D)

## 4.2 Evolution of the correlation functions, $\Omega$ and $\Delta\Omega$

In [ ]:
REFERENCE_4D = 0              # reference step for Omega: position in the series (0 = first)
SIGNED_4D = False             # True: Delta-Omega keeps its sign (net change)

for name in SMDS_4D:
    plot_curve_evolution(results_4d, steps_4d, name, save_as=os.path.join(OUT_4D, f'evolution_{name}.png'), ndim=3)

metrics_4d = evolution_metrics(results_4d['raw'], steps_4d, reference_index=REFERENCE_4D, signed=SIGNED_4D)
plot_omega(metrics_4d, save_as=os.path.join(OUT_4D, 'omega.png'))

save_table(stack_tables(results_4d['raw'], steps_4d), os.path.join(OUT_4D, 'smds_raw.csv'))
save_table(stack_tables(results_4d['scaled'], steps_4d), os.path.join(OUT_4D, 'smds_scaled.csv'))
save_table(metrics_4d, os.path.join(OUT_4D, 'omega.csv'))
metrics_4d

## 4.3 Evolution of the chord length

In [ ]:
plot_chord_evolution(results_4d['chord'], steps_4d, show=None, max_length=100, unit='voxels',
                     save_as=os.path.join(OUT_4D, 'chord_length.png'))

chord_table_4d = pd.DataFrame({'step': steps_4d,
                               'ell_via_s2': [result['ell_via_s2'] for result in results_4d['chord']],
                               'ell_direct': [result['ell_direct'] for result in results_4d['chord']]})
save_table(chord_table_4d, os.path.join(OUT_4D, 'chord_length.csv'))
chord_table_4d

## 4.4 Evolution of the Minkowski functionals

In [ ]:
mink_table_4d = pd.DataFrame(results_4d['minkowski'])
mink_table_4d.insert(0, 'step', steps_4d)

plot_columns(mink_table_4d, x='step', xlabel='time step', save_as=os.path.join(OUT_4D, 'minkowski.png'))
save_table(mink_table_4d, os.path.join(OUT_4D, 'minkowski.csv'))
mink_table_4d

# References

**Correlation functions**

- Torquato, S. (2002). *Random Heterogeneous Materials: Microstructure and Macroscopic Properties.* Springer, New York. (Ch. 2 for the definitions, Sec. 12.5 for sampling from digitised images.)
- Yeong, C. L. Y., & Torquato, S. (1998). Reconstructing random media. *Physical Review E*, 57, 495-506. https://doi.org/10.1103/PhysRevE.57.495
- Debye, P., Anderson, H. R., & Brumberger, H. (1957). Scattering by an inhomogeneous solid. II. The correlation function and its application. *Journal of Applied Physics*, 28, 679-683. https://doi.org/10.1063/1.1722830
- Berryman, J. G., & Blair, S. C. (1986). Use of digital image analysis to estimate fluid permeability of porous materials: Application of two-point correlation functions. *Journal of Applied Physics*, 60, 1930-1938. https://doi.org/10.1063/1.337245
- Torquato, S., Beasley, J. D., & Chiew, Y. C. (1988). Two-point cluster function for continuum percolation. *The Journal of Chemical Physics*, 88, 6540-6547. https://doi.org/10.1063/1.454440
- Jiao, Y., Stillinger, F. H., & Torquato, S. (2009). A superior descriptor of random textures and its predictive capacity. *Proceedings of the National Academy of Sciences*, 106, 17634-17639. https://doi.org/10.1073/pnas.0905919106
- Lu, B., & Torquato, S. (1992). Lineal-path function for random heterogeneous materials. *Physical Review A*, 45, 922-929. https://doi.org/10.1103/PhysRevA.45.922
- Torquato, S., & Lu, B. (1993). Chord-length distribution function for two-phase random media. *Physical Review E*, 47, 2950-2953. https://doi.org/10.1103/PhysRevE.47.2950

**Polytope functions and evolution metrics**

- Chen, P.-E., Xu, W., Chawla, N., Ren, Y., & Jiao, Y. (2019). Hierarchical n-point polytope functions for quantitative representation of complex heterogeneous materials and microstructural evolution. *Acta Materialia*, 179, 317-327. https://doi.org/10.1016/j.actamat.2019.08.045
- Chen, P.-E., et al. (2022). Quantifying microstructural evolution via time-dependent reduced-dimension metrics based on hierarchical n-point polytope functions. *Physical Review E*, 105, 025306. https://doi.org/10.1103/PhysRevE.105.025306

**Representative elementary volume**

- Bear, J. (1972). *Dynamics of Fluids in Porous Media.* Elsevier, New York.
- Amiri, H., et al. (2024). https://doi.org/10.1029/2024JH000178

**Minkowski functionals**

- Mecke, K. R. (2000). Additivity, convexity, and beyond: Applications of Minkowski functionals in statistical physics. In *Statistical Physics and Spatial Statistics*, Lecture Notes in Physics 554, 111-184. Springer.
- Legland, D., Kiêu, K., & Devaux, M.-F. (2007). Computation of Minkowski measures on 2D and 3D binary images. *Image Analysis & Stereology*, 26, 83-92. https://doi.org/10.5566/ias.v26.p83-92
- Armstrong, R. T., McClure, J. E., Robins, V., Liu, Z., Arns, C. H., Schlüter, S., & Berg, S. (2019). Porous media characterization using Minkowski functionals: Theories, applications and future directions. *Transport in Porous Media*, 130, 305-335. https://doi.org/10.1007/s11242-018-1201-4
- Boelens, A. M. P., & Tchelepi, H. A. (2021). QuantImPy: Minkowski functionals and functions with Python. *SoftwareX*, 16, 100823. https://doi.org/10.1016/j.softx.2021.100823

**Connected components and shape measurements**

- van der Walt, S., et al. (2014). scikit-image: image processing in Python. *PeerJ*, 2, e453. https://doi.org/10.7717/peerj.453
- Lorensen, W. E., & Cline, H. E. (1987). Marching cubes: A high resolution 3D surface construction algorithm. *ACM SIGGRAPH Computer Graphics*, 21, 163-169. https://doi.org/10.1145/37402.37422
- Wadell, H. (1935). Volume, shape, and roundness of quartz particles. *The Journal of Geology*, 43, 250-280. https://doi.org/10.1086/624298

**Software**

- Lam, S. K., Pitrou, A., & Seibert, S. (2015). Numba: A LLVM-based Python JIT compiler. *Proceedings of the Second Workshop on the LLVM Compiler Infrastructure in HPC.* https://doi.org/10.1145/2833157.2833162